# Fine-Grained Pet Breed Classification

## Oxford-IIIT Pet Dataset

This project focuses on building an end-to-end Computer Vision system
for fine-grained pet breed classification using Deep Learning.

# 🏛️ 1. Project Overview

## 1.1 Problem Statement

Given an input pet image, the model must predict its breed from
37 possible cat and dog breed classes.

```text
Input Image
     ↓
Computer Vision Model
     ↓
Predicted Breed
```

This is a multi-class fine-grained image classification problem.

## 1.2 Project Objectives

The main objectives are to:

- Build a complete Computer Vision pipeline.
- Analyze and prepare image data.
- Train a CNN baseline.
- Apply Transfer Learning.
- Compare different models.
- Analyze model errors.
- Explain predictions using Grad-CAM.
- Deploy the final model through an API.

## 1.3 Why Fine-Grained Classification?

Fine-grained classification requires distinguishing between visually
similar classes.

Different pet breeds may share similar:

- Fur patterns
- Face structures
- Body shapes
- Colors
- Ear shapes

Therefore, the model must learn subtle visual features rather than
only broad differences between cats and dogs.

## 1.4 Dataset Overview

The Oxford-IIIT Pet Dataset contains 7,349 images covering
37 categories of cats and dogs.

The dataset contains 25 dog breeds and 12 cat breeds, with
approximately 200 images per class.

The original dataset also provides breed labels, head bounding boxes,
and pixel-level segmentation annotations.

## 1.5 Machine Learning Task

The project is formulated as a supervised multi-class classification
problem.

```text
Input:
Pet Image

Target:
Breed Class

Output:
Predicted Breed + Class Probabilities
```

The model will learn the mapping between visual patterns in the
input image and the corresponding breed label.

## 1.6 Success Criteria

The final system should:

- Achieve strong classification performance.
- Generalize to unseen test images.
- Perform well across different breed classes.
- Provide interpretable predictions.
- Support reliable inference.
- Be suitable for deployment.

# 2. Environment & Configuration

A reproducible Computer Vision project requires a consistent software
and hardware environment.

Before loading the dataset, we will configure:

- Required libraries
- Reproducibility settings
- Computing device
- Project paths
- Random seeds

## 2.1 Import Libraries

We will use a small set of libraries throughout the project:

- PyTorch for Deep Learning
- Torchvision for Computer Vision utilities
- NumPy for numerical operations
- Pandas for structured data
- Matplotlib and Seaborn for visualization
- PIL for image loading
- Pathlib for project paths

In [1]:
# ============================================================
# Import Core Libraries
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader

## 2.2 Reproducibility

Machine Learning experiments can produce different results when random
operations are not controlled.

Reproducibility means that running the same experiment with the same
configuration should produce highly consistent results.

We will control the main sources of randomness used in the project.

## 2.3 Device Configuration

Deep Learning models can be trained on:

- CPU
- GPU

GPU training is significantly more suitable for CNN and Transfer
Learning experiments because image models involve large numbers of
tensor operations.

We will automatically select CUDA when it is available and fall back
to CPU otherwise.

In [2]:
# ============================================================
# Check the Available Computing Device
# ============================================================

# Use the GPU when CUDA is available.
# Otherwise, fall back to CPU.

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)


# ------------------------------------------------------------
# Display GPU information when available
# ------------------------------------------------------------

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA Version:",
        torch.version.cuda
    )
else:
    print("CUDA is not available. Using CPU.")

Selected device: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
CUDA Version: 13.0


## 2.4 Project Configuration

Project configuration should be centralized instead of being repeated
throughout the notebook.

We will define the main paths and experiment parameters once and reuse
them throughout the project.

In [3]:
# ============================================================
# Project Configuration
# ============================================================

# Define the project root directory.
# The exact location depends on where the notebook is stored.

PROJECT_ROOT = Path(r"C:\Users\mahmu\OneDrive\Desktop\Fine-Grained Pet Breed Classification")

# Dataset directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
IMAGE_DIR = PROCESSED_DIR / "images"
ANNOTATION_DIR = PROCESSED_DIR / "annotations"

# Experiment directories
REPORT_DIR = PROJECT_ROOT / "reports"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

# Main experiment configuration
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 37
RANDOM_SEED = 42

print("Project root:", PROJECT_ROOT.resolve())
print("Image directory:", IMAGE_DIR.resolve())
print("Annotation directory:", ANNOTATION_DIR.resolve())

Project root: C:\Users\mahmu\OneDrive\Desktop\Fine-Grained Pet Breed Classification
Image directory: C:\Users\mahmu\OneDrive\Desktop\Fine-Grained Pet Breed Classification\data\processed\images
Annotation directory: C:\Users\mahmu\OneDrive\Desktop\Fine-Grained Pet Breed Classification\data\processed\annotations


## 2.5 Random Seeds

We will set random seeds for Python, NumPy, and PyTorch.

This reduces unnecessary variation between repeated experiments and
makes model development easier to reproduce.

In [4]:
# ============================================================
# Set Random Seeds
# ============================================================

import random

# Seed Python's built-in random module.
random.seed(RANDOM_SEED)

# Seed NumPy.
np.random.seed(RANDOM_SEED)

# Seed PyTorch on the CPU.
torch.manual_seed(RANDOM_SEED)

# Seed all CUDA devices when a GPU is available.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Random seed:", RANDOM_SEED)

Random seed: 42


## 2.6 Environment Check

Before continuing, we verify the main software components used by the
project.

This helps us detect environment problems before starting the
experiments.

In [5]:
# ============================================================
# Verify the Main Environment
# ============================================================

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

print("\nDevice:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.12.0+cu130
Torchvision version: 0.27.0+cu130
NumPy version: 2.4.5
Pandas version: 3.0.3

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


# 3. Dataset Loading & Exploration

This section introduces the Oxford-IIIT Pet dataset and verifies that the image files and annotation files are correctly loaded before performing any analysis.

## 3.1 Load Dataset

The Oxford-IIIT Pet dataset contains pet images organized by breed.

The processed dataset includes:

- Image files
- Training and validation annotations
- Test annotations
- Class labels
- Additional annotation files

We will first load the annotation files and use them to connect each image with its corresponding breed label.

In [6]:
# ============================================================
# Load Oxford-IIIT Pet Annotation Files
# ============================================================

# The Oxford-IIIT Pet annotations contain the information
# needed to connect image files with their corresponding labels.

# Load the training/validation annotation file.
trainval_file = ANNOTATION_DIR / "trainval.txt"

# Load the test annotation file.
test_file = ANNOTATION_DIR / "test.txt"

# Read the annotation files line by line.
trainval_lines = trainval_file.read_text().splitlines()
test_lines = test_file.read_text().splitlines()

# Display the number of annotation records.
print("Train/Validation records:", len(trainval_lines))
print("Test records:", len(test_lines))

# Inspect a few records to understand the annotation format.
print("\nSample Train/Validation records:")
for line in trainval_lines[:5]:
    print(line)

print("\nSample Test records:")
for line in test_lines[:5]:
    print(line)

Train/Validation records: 3680
Test records: 3669

Sample Train/Validation records:
Abyssinian_100 1 1 1
Abyssinian_101 1 1 1
Abyssinian_102 1 1 1
Abyssinian_103 1 1 1
Abyssinian_104 1 1 1

Sample Test records:
Abyssinian_201 1 1 1
Abyssinian_202 1 1 1
Abyssinian_204 1 1 1
Abyssinian_205 1 1 1
Abyssinian_206 1 1 1


## 3.2 Dataset Structure

The annotation files follow a structured format where each row describes one image.

Each record contains four fields:

- Image ID
- Class ID
- Species ID
- Breed ID

This structure allows us to connect each image file with its corresponding class and species information.

In [7]:
# ============================================================
# Inspect the Annotation Structure
# ============================================================

# Split the first annotation record into individual fields.
sample_fields = trainval_lines[0].split()

# Display the raw record.
print("Raw record:")
print(trainval_lines[0])

# Display each field separately.
print("\nParsed fields:")

print("Image ID :", sample_fields[0])
print("Class ID :", sample_fields[1])
print("Species ID:", sample_fields[2])
print("Breed ID :", sample_fields[3])

# Verify the number of fields in the annotation format.
print("\nNumber of fields:", len(sample_fields))

Raw record:
Abyssinian_100 1 1 1

Parsed fields:
Image ID : Abyssinian_100
Class ID : 1
Species ID: 1
Breed ID : 1

Number of fields: 4


## 3.3 Number of Samples

The dataset contains separate annotation records for the training/validation
and test partitions.

The total number of samples is obtained by combining both partitions.

In [8]:
# ============================================================
# Count Dataset Samples
# ============================================================

# Count the number of records in each dataset partition.
trainval_count = len(trainval_lines)
test_count = len(test_lines)

# Calculate the total number of samples.
total_samples = trainval_count + test_count

print("Train/Validation samples:", trainval_count)
print("Test samples:", test_count)
print("Total samples:", total_samples)

Train/Validation samples: 3680
Test samples: 3669
Total samples: 7349


## 3.4 Number of Classes

The dataset contains multiple pet breed classes.

We will determine the number of unique classes directly from the annotation files rather than assuming the documented number.

The class identifier is stored in the second field of each annotation record.

In [9]:
# ============================================================
# Count the Number of Classes
# ============================================================

# Combine both annotation partitions.
all_annotation_lines = trainval_lines + test_lines

# Extract the class ID from each annotation record.
class_ids = [
    int(line.split()[1])
    for line in all_annotation_lines
]

# Find the unique class IDs.
unique_class_ids = sorted(set(class_ids))

# Count the number of unique classes.
num_classes = len(unique_class_ids)

print("Number of unique classes:", num_classes)
print("Class IDs:", unique_class_ids)

Number of unique classes: 37
Class IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
